In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
import astropy.units as u
import statmorph

import utils.data as datutils
import utils.image as imutils
import utils.plots as plots

In [ ]:
def morph_a():
    morphs_list = []
    for id in np.arange(1, 21):
        try:
            file = fits.open(
                f'data/gadgetx3k_20/maps/bcg_{str(id).zfill(4)}_125_1.fits')
        except FileNotFoundError:
            continue
        print(f"Processing region {id}...")
        image = file[0].data
        center = (int(len(image[1])//2), int(len(image[0])//2))
        # r1 = 50*u.kpc
        # r1 = real2pix(r=r1, map=image)
        # r2 = 1*u.Mpc
        # r2 = real2pix(r=r2, map=image)
        segmap = imutils.annular_mask(image, center, r1=20, r2=409)

        fig, axs = plt.subplots(1, 1)
        plots.display_img(image, axs, mask=segmap)
        plt.show()
        morph = statmorph.source_morphology(image, segmap, gain=2.25)
        print(f'-------region {id} done-------')
        morphs_list.append(morph[0])

    sm_df = datutils.create_morph_df(morphs_list,
                                     name='results/rin_50kpc_rout_1Mpc.csv',
                                     save=False)

    return

In [ ]:
x = [s for s in np.arange(5)]
x.append(9)
print(x)

In [ ]:
def morph_c(map_dir):
    finished_ids = [s for s in np.arange(133)]
    morphs_list = []
    files_list = sorted(os.listdir(map_dir))
    for file in files_list:
        id = datutils.find_id(file)
        if id in finished_ids:
            continue
        else:
            print(f"Processing region {id}")
            finished_ids.append(id)

        map = datutils.load_map(file, map_dir)
        if map is None:
            print(f"Skipping region {id}: no map")
            continue

        radius = datutils.real2pix(r=1*u.Mpc, map=map)
        center = (int(len(map[1])//2), int(len(map[0])//2))
        segmap = imutils.circular_segmap(map, center, radius=radius)
        morph = statmorph.source_morphology(map, segmap, gain=2.25)
        morphs_list.append((id, morph[0]))

        print(f'{file} done')

        if len(morphs_list) % 20 == 0:
            sm_df = datutils.create_morph_df(morphs_list,
                                            name=f'results/gadgetx3k_{len(morphs_list)}_1Mpc_oops.csv',
                                            save=True)

    sm_df = datutils.create_morph_df(morphs_list, name=f'results/gadgetx3k_{len(morphs_list)}_1Mpc.csv', save=True)

    return

In [ ]:
morph_c('data/gadgetx3k_20/maps/OLD_ICs/')